# Dimensional Model Design

## Selected Business Processes

The main business processes selected from the OLTP system are:

1. Film Rental Transactions
2. Payment Transactions

These processes are important because they capture the main operational activities of the movie rental business and provide measurable analytical insights.

---

## Fact Tables

### FactRental

**Business Process:** Film rental transactions

**Grain:** One row per rental transaction

**Measures:**
- rental_duration_days (actual days the film was kept)
- is_late (1 = late return, 0 = on time)
- late_days (number of days overdue)

**Foreign Keys:**
- date_key
- customer_key
- film_key
- store_key
- staff_key

---

### FactPayment

**Business Process:** Payment transactions

**Grain:** One row per payment transaction

**Measures:**
- amount (payment amount in USD)

**Foreign Keys:**
- date_key
- customer_key
- store_key
- staff_key

---

## Dimension Tables

### DimDate
Stores date-related attributes for trend analysis.

Attributes:
- date_key
- full_date
- day
- month
- quarter
- year

Source Table:
- rental
- payment

---
### DimCustomer
Attributes:
- customer_key
- customer_id
- full_name
- email
- address
- city
- country
- store_id
- active

Source Tables:
- customer
- address
- city
- country

---

### DimFilm

Attributes:
- film_key
- film_id
- title
- release_year
- rental_rate
- length

Source Table:
- film

---

### DimCategory

Attributes:
- category_key
- category_name

Source Tables:
- category
- film_category

### DimStore
Attributes:
- store_key
- store_id
- address
- city
- country

Source Tables:
- store
- address
- city
- country

---

### DimStaff

Attributes:
- staff_key
- staff_id
- full_name

Source Table:
- staff

---
### DimLocation
DimLocation centralizes geographical attributes from 
address, city, and country tables for location-based analysis.

It supports analytical questions such as:
- Which cities generate the highest rental activity?
- Which countries have the most active customers?

Note: DimLocation is prepared as a supporting dimension
for future analytical expansion. Customer location details
are currently embedded in DimCustomer for direct analysis.

---

### DimLanguage
Language attribute is integrated within DimFilm.
Although a language table exists in the OLTP system,
all 1,000 films share the same language (English),
therefore language was embedded in DimFilm as an
attribute rather than a separate dimension.

---

### DimActor
DimActor stores actor information extracted from the actor table.

Attributes:
- actor_key
- actor_id
- full_name

Source Table:
- actor

Note: Since films and actors have a many-to-many relationship
through the film_actor bridge table, DimActor was prepared
for future analytical expansion. Full integration would require
a bridge table linking DimActor to fact_rental through film_actor.


---

## Schema Type

A Star Schema is selected because it simplifies analytical queries and improves reporting performance by organizing data around central fact tables connected to descriptive dimensions.

In [5]:
import pymysql
import pandas as pd

conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root123",
    database="sakila",
    connect_timeout=5
)
cursor = conn.cursor()
print("Connected ✅")

Connected ✅


In [6]:
# dim_actor
cursor.execute("""
    SELECT 
        a.actor_id,
        CONCAT(a.first_name, ' ', a.last_name) as full_name
    FROM actor a
""")
rows = cursor.fetchall()
dim_actor = pd.DataFrame(rows, columns=['actor_id', 'full_name'])
dim_actor.insert(0, 'actor_key', range(1, len(dim_actor) + 1))

print("dim_actor READY!")
print(f" Rows: {len(dim_actor)}")
print(dim_actor.head(3))

# dim_location
cursor.execute("""
    SELECT DISTINCT
        ci.city_id,
        ci.city,
        co.country
    FROM city ci
    JOIN country co ON ci.country_id = co.country_id
""")
rows = cursor.fetchall()
dim_location = pd.DataFrame(rows, columns=['city_id', 'city', 'country'])
dim_location.insert(0, 'location_key', range(1, len(dim_location) + 1))

print("\ndim_location Ready!")
print(f"Rows: {len(dim_location)}")
print(dim_location.head(3))

dim_actor READY!
 Rows: 200
   actor_key  actor_id         full_name
0          1         1  PENELOPE GUINESS
1          2         2     NICK WAHLBERG
2          3         3          ED CHASE

dim_location Ready!
Rows: 600
   location_key  city_id    city      country
0             1      251   Kabul  Afghanistan
1             2       59   Batna      Algeria
2             3       63  Béchar      Algeria


In [7]:
import pymysql
import pandas as pd

conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root123",
    database="sakila",
    connect_timeout=5
)
cursor = conn.cursor()
print("Connected ✅")

Connected ✅


In [8]:
import os
path = r'C:\Users\User\OneDrive\Desktop\movie_rental_dw\data\\'
dim_actor.to_csv(path + 'dim_actor.csv', index=False)
dim_location.to_csv(path + 'dim_location.csv', index=False)
print("Saved ✅")

Saved ✅
